In [14]:
import boto3, botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile, base64
from datetime import date
from dotenv import load_dotenv


from misc import load_from_yaml, save_to_yaml
import iam, s3, lf, rds, aws_networking, ec2

load_dotenv(".env")
# boto3.setup_default_session(profile_name="AMominNJ")

True

In [15]:
ACCOUNT_ID = os.environ["AWS_ACCOUNT_ID_ROOT"]
REGION = os.environ["AWS_DEFAULT_REGION"]
VPC_ID = os.environ["AWS_DEFAULT_VPC"]
SECURITY_GROUP_ID = os.environ["AWS_DEFAULT_SG_ID"]
SUBNET_IDS = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID = SUBNET_IDS[0]
AWS_INSTANCE_ID_JMASTER = os.environ["AWS_INSTANCE_ID_JMASTER"]
# AWS_INSTANCE_ID_JAGENT = os.environ["AWS_INSTANCE_ID_JAGENT"]
AWS_DEFAULT_IMAGE_ID = os.environ["AWS_DEFAULT_IMAGE_ID"]
AWS_DEFAULT_KEY_PAIR_NAME = os.environ["AWS_DEFAULT_KEY_PAIR_NAME"]
AWS_DEFAULT_INSTANCE_TYPE = os.environ["AWS_DEFAULT_INSTANCE_TYPE"]

In [16]:
acm_client = boto3.client("acm", region_name=REGION)
acmpca_client = boto3.client("acm-pca", region_name=REGION)
route53_client = boto3.client("route53", region_name=REGION)

### ACM -> Tested Successfully

-   [boto3 Doc: ACM](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/acm.html) | [Troubleshoot DNS validation problems](https://docs.aws.amazon.com/acm/latest/userguide/troubleshooting-DNS-validation.html)
-   [AWS IAM Roles Anywhere with OpenSSL](https://www.youtube.com/watch?v=aIX9by2uEgU)
-   [Masterclass in openSSL](https://www.youtube.com/watch?v=d8OpUcHzTeg&t=231s)
-   [OpenSSL Certification Authority (CA) on Ubuntu Server](https://www.youtube.com/watch?v=oCl0gzLPPMI)
-   [Intro to Digital Certificates](https://www.youtube.com/watch?v=qXLD2UHq2vk)
-   [Create CSR & Install SSL Certificate (OpenSSL)](https://www.digicert.com/kb/csr-ssl-installation/apache-openssl.htm#ssl_certificate_install)
-   [Cacert | Trust Store | Key Store in 3 minutes](https://www.youtube.com/watch?v=1I9lQ42SnLA)


In [ ]:
zones = route53_client.list_hosted_zones()["HostedZones"]
for zone in zones:
    if zone["Name"] == "harnesstechtx.com.": hosted_zone_id = zone["Id"].split("/")[-1]
print(hosted_zone_id)

Z04555692B7PI94BFJEBI


In [ ]:
domain_name = "harnesstechtx.com"
subject_alternative_names = ["www.harnesstechtx.com", "sub.harnesstechtx.com"]

Z04555692B7PI94BFJEBI


In [ ]:
response = acm_client.request_certificate(
    DomainName=domain_name,
    SubjectAlternativeNames=subject_alternative_names,
    ValidationMethod="DNS",  # Can be "DNS" or "EMAIL"
    Options={"CertificateTransparencyLoggingPreference": "ENABLED"},
    Tags=[{"Key": "Environment", "Value": "QA"}],
)


# Get the certificate ARN
certificate_arn = response["CertificateArn"]
print(f"Certificate request initiated. ARN: {certificate_arn}")

Certificate request initiated. ARN: arn:aws:acm:us-east-1:381492255899:certificate/75b2fd3a-fe03-454e-906b-e10a799a008a


- Wait for ACM to generate validation records

In [ ]:
print("Waiting for ACM to generate DNS validation records...")
time.sleep(10)  # Give AWS some time to generate records


- Fetch DNS validation details

In [ ]:
certificate_details = acm_client.describe_certificate(CertificateArn=certificate_arn)
dns_records = []
for domain_validation in certificate_details["Certificate"]["DomainValidationOptions"]:
    if "ResourceRecord" in domain_validation:
        dns_records.append(domain_validation["ResourceRecord"])

In [ ]:
print(dns_records)

[{'Name': '_524fe47125fcbec96e26e8ee40dd2e77.harnesstechtx.com.', 'Type': 'CNAME', 'Value': '_afc6c5fd0f07576f49c0ebb8aebc6217.zfyfvmchrl.acm-validations.aws.'}, {'Name': '_836c8f27f8002164008547c2e207f52e.www.harnesstechtx.com.', 'Type': 'CNAME', 'Value': '_cbef82367961f09b20302eeb8b8ee19f.xlfgrmvvlj.acm-validations.aws.'}, {'Name': '_524fe47125fcbec96e26e8ee40dd2e77.harnesstechtx.com.', 'Type': 'CNAME', 'Value': '_afc6c5fd0f07576f49c0ebb8aebc6217.zfyfvmchrl.acm-validations.aws.'}]


- Add DNS validation records to Route 53

In [ ]:
changes = []
for record in dns_records:
    changes.append(
        {
            "Action": "UPSERT",
            "ResourceRecordSet": {
                "Name": record["Name"],
                "Type": record["Type"],
                "TTL": 300,
                "ResourceRecords": [{"Value": record["Value"]}],
            },
        }
    )

print(changes)

[{'Action': 'UPSERT', 'ResourceRecordSet': {'Name': '_524fe47125fcbec96e26e8ee40dd2e77.harnesstechtx.com.', 'Type': 'CNAME', 'TTL': 300, 'ResourceRecords': [{'Value': '_afc6c5fd0f07576f49c0ebb8aebc6217.zfyfvmchrl.acm-validations.aws.'}]}}, {'Action': 'UPSERT', 'ResourceRecordSet': {'Name': '_836c8f27f8002164008547c2e207f52e.www.harnesstechtx.com.', 'Type': 'CNAME', 'TTL': 300, 'ResourceRecords': [{'Value': '_cbef82367961f09b20302eeb8b8ee19f.xlfgrmvvlj.acm-validations.aws.'}]}}, {'Action': 'UPSERT', 'ResourceRecordSet': {'Name': '_524fe47125fcbec96e26e8ee40dd2e77.harnesstechtx.com.', 'Type': 'CNAME', 'TTL': 300, 'ResourceRecords': [{'Value': '_afc6c5fd0f07576f49c0ebb8aebc6217.zfyfvmchrl.acm-validations.aws.'}]}}]


In [ ]:
# Adding DNS validation records to Route 53
route53_client.change_resource_record_sets(
    HostedZoneId=hosted_zone_id,
    ChangeBatch={
        "Comment": "Adding DNS validation records for ACM",
        "Changes": changes,
    },
)

- Wait for ACM validation

In [ ]:
while True:
    certificate_status = acm_client.describe_certificate(CertificateArn=certificate_arn)["Certificate"]["Status"]
    print(f"Current certificate status: {certificate_status}")

    if certificate_status == "ISSUED":
        print(f"Certificate successfully validated and issued: {certificate_arn}")
        break
    elif certificate_status == "FAILED":
        print("Certificate validation failed. Check AWS ACM console.")
        break

    time.sleep(30)  # Wait before checking again

print("Script execution complete.")


#### Delete Resources

In [ ]:
acm_client.delete_certificate(CertificateArn=certificate_arn)

{'ResponseMetadata': {'RequestId': 'bcdeedbe-cdca-4d77-a435-b4ac68aa91cc',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'bcdeedbe-cdca-4d77-a435-b4ac68aa91cc',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '0',
   'date': 'Thu, 01 May 2025 13:42:07 GMT'},
  'RetryAttempts': 0}}